# Evaluation: Benchmarks, Evals, LM Harness

## 问题描述

MMLU基准在2020发布，如今领先的模型分数区间在3分以内，这是统计误差导致的，而非模型真正的能力。

与此同时，模型在拿高分的同时又在一些简单的任务上出错。

所以基准测试性能和实际可靠性之间的间隔就是模型评估的核心问题。

## 基本概念

有三种评估种类，每种的开销和信息都不同：

### 基准测试

拿模型跑基准测试然后得到一个分数。好处在于所有人执行的测试都是相通的，所以你可以横向比较模型。坏处在于，模型可以刷分，但是不代表能力上升了。

#### 失效原因
1. 数据污染。模型的训练语料中包含基准测试题库。
2. 教会模型做题。
3. 饱和。从87分到89分可能只是答对了几个冷门题目，模型的能力并没有上升。

### 自定义评估

更适合你自己的使用场景。输入、预期输出以及打分标准都由你自己来决定。创建它很贵，但是它同时是衡量你的产品性能的唯一标准。

### 人力评估

花钱找一堆标注员来评估模型输出。

### 困惑度

用来衡量模型对某个序列词元出现的惊讶程度。
```
PPL = exp(-1 * sum(log P(token_i | context) / N))
```

直观的理解困惑度就是，当困惑度为k时，模型对于下一词元的不确定性就好像从k个候选中挑出了它一样。

困惑度有用，但也有盲点。模型在擅长预测通用模式但是在罕见的模式长表现很差时，也能拿到较低的困惑度评分。另外，它也反映不了对指令遵循、推理或者事实准确度等指标。

### 大模型作为评委

使用一个牛逼的大模型来评估较弱模型的输入。原因很直观：大模型的标注成本低廉，而且现在已经与人类的一致性可能达到80%以上。

比较重要的是打分提示词，你需要给出结果化的提示词，而不是“请为答案打分”这种模糊不清的提示词，让打分结果连贯、可复现。

失败模式包含：1. 偏向于第一条回复；2. 偏向长回复；3. 偏向于自家模型。
缓解手段：打乱顺序，归一化长度，使用不同的模型进行评估。

### ELO 打分

同一个Prompt喂给不同的模型，由人（或LLM）挑选出更好的那个。然后计算模型的ELO比率。


# 开始编码

In [1]:
class EvalCase:
    def __init__(self, input_text, expected, meta_data=None):
        self.input_text = input_text
        self.expected = expected
        self.meta_data = meta_data or {}

class EvalSuite:
    def __init__(self, name, cases, scorers):
        self.name = name
        self.cases = cases
        self.scorers = scorers

    def run(self, model_fn):
        result = []
        for case in self.cases:
            predictions = model_fn(case.input_text)
            scores = {}
            for scorer_name, scorer_fn in self.scorers.items():
                scores[scorer_name] = scorer_fn(predictions, case.expected)
            result.append({
                "input": case.input_text,
                "expected": case.expected,
                "predictions": predictions,
                "scores": scores
            })
        return result
    

In [ ]:
def exact_match(prediction, expected):
    return 1.0 if prediction.strip().lower() == expected.strip().lower() else 0.0



def token_f1(prediction, expected):
    pred_tokens = set(prediction.lower().split())
    expected_tokens = set(expected.lower().split())
    if not pred_tokens or not expected_tokens:
        return 0.0
    intersection = pred_tokens.intersection(expected_tokens)
    precision = len(intersection) / len(pred_tokens)
    recall = len(intersection) / len(expected_tokens)
    return 2 * precision * recall / (precision + recall)

def llm_judge(prediction, expected):
    prompt = f"""
    你是一个专业的评估者，现在需要评估两个模型对于同一个输入的回答。
    请根据以下标准对两个回答进行打分：
    1. 准确性：回答是否准确地回答了问题。
    2. 完整性：回答是否完整地回答了问题。
    3. 清晰性：回答是否清晰地回答了问题。
    
    请给出你的打分，并给出打分理由。
    
    输入：
    {input}

    回答1：
    {prediction}
    
    回答2：
    {expected}
    """
    pass
    


In [ ]:
class ELOTracker:
    def __init__(self, k=32, initial_rating=500):
        self.ratings = {}
        self.k = k
        self.initial_rating = initial_rating
        self.history = []

    def expcted_score(self, rating_a, rating_b):
        return 1 / (1 + 10**((rating_b - rating_a) / 400))

    def _ensure_player(self, player_name):
        if player_name not in self.ratings:
            self.ratings[player_name] = self.initial_rating

    def record_match(self, player_a, player_b, outcome):
        self._ensure_player(player_a)
        self._ensure_player(player_b)

        ea = self.expcted_score(self.ratings[player_a], self.ratings[player_b])
        eb = 1 - ea

        if outcome == "a":
            sa, sb = 1.0, 0.0
        elif outcome == "b":
            sa, sb = 0.0, 1.0
        else:
            sa, sb = 0.5, 0.5

        self.ratings[player_a] += self.k * (sa - ea)
        self.ratings[player_b] += self.k * (sb - eb)

        self.history.append({
            "player_a": player_a,
            "player_b": player_b,
            "outcome": outcome,
            "rating_a": round(self.ratings[player_a], 1),
            "rating_b": round(self.ratings[player_b], 1)
        })

    def get_leaderboard(self):
        return sorted(self.ratings.items(), key=lambda x: x[1], reverse=True)
    
        
        

In [ ]:
import numpy as np

def perplexity(log_probs):
    if not log_probs:
        return float('inf')
    avg_neg_log_prob = -np.mean(log_probs)
    return float(np.exp(avg_neg_log_prob))